## Import files

In [1]:
# Import data processing modules
import pandas as pd # used throughout the code
import numpy as np # used throughout the code
import country_converter as coco # used for converting country names to ISO codes
from sklearn.impute import KNNImputer # used for imputation of values for missing country-climate zone combinations
#import scipy as sp # used for correlation analysis after disaggregation


# Import visualization modules
import matplotlib.pyplot as plt # used for simple bar charts and as basis for plotly and seaborn library
import plotly.express as px # used for all stacked bar charts
import plotly.graph_objects as go # used for marimekko plot and for annotating stacked bar charts
import seaborn as sns # used for scatterplot and kdeplot in data and sensitivity analysis section after disaggregation
from textwrap import fill # used for scatterplot in data analysis section after disaggregation

# Display options
pd.options.display.float_format = '{:,.2f}'.format #limits printed decimal points to two
#pd.reset_option('^display.', silent=True) #option to reset the previous display option

In [2]:
urbanity = False
climate_calc = False #set this to true to reproduce climate zone with NUTS region matching

if climate_calc == True:
    import geopandas as gpd
    import xarray as xr #used to open netcdf climate file

    from matplotlib.colors import ListedColormap
    from matplotlib import colormaps
    from matplotlib.gridspec import GridSpec
    from matplotlib import rcParams

    #%% Global plotting settings
    rcParams['font.family'] = 'Bitstream Vera Sans'
    rcParams['font.size'] = 10

#TODOS
- test how region_nuts compares to region_bld in absolute terms for region_gea
- calculate population per nuts3 and urbanity
- calculate building stock per nuts3 and urbanity
- tenure

- material intensity
- change the model to include unoccupied dwellings for material calculation

## Add Approach: add another column with NUTS3 labels to regions label file

In [3]:
#Functions

def nuts3code_to_region_nuts(input):
       input = pd.merge(input,code_to_region_nuts) #adding region labels
       input.drop(['NUTS-3 Code'], axis=1, inplace=True)
       return input

def all_rows_contained(df1, df2):
    """Check if all rows in df1 are contained in df2"""
    merged = df1.merge(df2, how='left', indicator=True)
    return (merged['_merge'] == 'both').all()

### Expanding the index

In [4]:
# Region Labels Detailed: creating labels that include an extra level for NUTS3 regions

#importing NUTS labels
if urbanity == True:
    usecols = ['Country code', 'NUTS-3 Code', 'Urban-Rural typology']
else:
    usecols = ['Country code', 'NUTS-3 Code']
nuts_lab = pd.read_excel('data/NUTS2021-NUTS2024.xlsx', sheet_name = 'NUTS-3 Typologies', header=0, usecols=usecols)
nuts_lab['iso3'] = coco.convert(names=nuts_lab['Country code'], to='ISO3')

#importing region labels
region_lab = pd.read_csv('2025_EU/input_others/regions_R61.csv')
region_lab['iso3']=region_lab.region_bld.str[-3:]

#appending NUTS labels to region labels where available
region_nuts_lab = pd.merge(nuts_lab, region_lab, on='iso3',how='left') #'outer'
region_nuts_lab['region_nuts'] = [str(x) + '-' + str(y) for x, y in zip(region_nuts_lab['region_bld'], region_nuts_lab['NUTS-3 Code'])]
region_nuts_lab['region_nuts'] = region_nuts_lab['region_nuts'].str.strip('-nan')

#cleaning df and harmonising with input dataframes
if urbanity == True:
    region_nuts_lab = region_nuts_lab.replace({'predominantly urban':'urb', 'intermediate':'urb', 'predominantly rural':'rur'})
    region_nuts_lab = region_nuts_lab.rename({'Urban-Rural typology':'urt'}, axis=1)
    
    #adding rural options to LUX, MLT, CYP
    region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('LUX')].assign(urt='rur'), ignore_index=True)
    region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('MLT')].assign(urt='rur'), ignore_index=True)
    region_nuts_lab = region_nuts_lab.append(region_nuts_lab[region_nuts_lab['region_nuts'].str.contains('CYP')].assign(urt='rur'), ignore_index=True)
    region_nuts_lab_urb = region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1)

    region_nuts_lab.drop(['urt'], axis=1, inplace=True)
    region_nuts_lab.drop_duplicates(inplace=True)

#define index to use when importing data
code_to_region_nuts = region_nuts_lab[['NUTS-3 Code', 'region_nuts', 'region_bld']] #input to the function nuts3code_to_region_nuts()

#export
region_nuts_lab = region_nuts_lab.drop(['iso3','NUTS-3 Code', 'Country code'], axis=1)
region_nuts_lab.to_csv('2025_EU/input_resid/regions_R61_nuts.csv', index=False)
region_nuts_lab.to_csv('2025_EU/input_others/regions_R61_nuts.csv', index=False)


#checks
print('Missing values: ',region_nuts_lab.isna().sum().sum())
print('Duplicates: ',region_nuts_lab.duplicated().sum().sum())
print('NUTS3 regions: ',len(region_nuts_lab.region_nuts.unique()))
print('Index still contained: ', region_lab.drop('iso3', axis=1)[region_lab.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True)\
      .equals(region_nuts_lab.drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

Missing values:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [5]:
#Climate Zones: overlaying NUTS regions with matching climate zones

if climate_calc == True:
       #import climate zones file
       ds = xr.open_dataset('data/climate_zones.nc')
       df = ds.to_dataframe()

       #assigning literal climate zone names to match with the clim variable
       df = df.combined
       climate_zones_names = pd.read_csv('data/climate_zones_names.csv', header=None)
       climate_zones_names.columns = ['combined', 'clim']
       climate_zones_names = climate_zones_names.iloc[1:]
       replacement_map = pd.Series(climate_zones_names.clim.values, index=climate_zones_names.combined).to_dict()
       df = df.replace(replacement_map)
       df.to_csv('data/climate_zones.csv')

       # merge climate zones with NUTS regions, assigning each NUTS region the climate zone that is most common to the area, in case of equal split, one of the zones is selected randomly
       climate = pd.read_csv('data/climate_zones.csv')
       gdf = gpd.GeoDataFrame(climate,geometry=gpd.points_from_xy(climate.lon,climate.lat)).drop(['lat', 'lon'], axis=1)
       nuts = gpd.read_file('data/NUTS_RG_20M_2024_3035.gpkg') #need to match the coordinate systems between gdf and nuts
       nuts = nuts.loc[nuts['NUTS_ID'].str.len() == 5] #only NUTS3 level
       gdf = gdf.set_crs(epsg=4326)
       gdf = gdf.to_crs(epsg=3035)
       nuts_climate = nuts.sjoin_nearest(gdf, how='left')
       nuts_climate = nuts_climate.drop(['index_right', 'geometry', 'LEVL_CODE', 'CNTR_CODE', 'NAME_LATN', 'NUTS_NAME',
              'MOUNT_TYPE', 'URBN_TYPE', 'COAST_TYPE'], axis=1).groupby(by=['NUTS_ID']).agg(lambda x: pd.Series.mode(x)[0]).reset_index()
       nuts_climate.columns = ['NUTS-3 Code', 'clim']
       nuts_climate.to_csv('data/climate_nuts.csv', index=False)

input = pd.read_csv('data/climate_nuts.csv', index_col=[0])

input = nuts3code_to_region_nuts(input)

#import file
filename= 'climatic_zones_rev'
input_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

#harmonize index
input_nuts = pd.merge(input_original.drop('clim', axis=1).drop_duplicates(), input, on=['region_bld'], how='right')

#export
if input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT']).sum()>0:
       input_nuts = input_nuts[~input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
else:
       print('no double entries')
input_nuts.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

#index
region_nuts_clim = input_nuts.copy()

#checks
print('Climate zones missing from EU-NUTS dataset:', set(input_original.clim.unique()) - set(input_nuts.clim.unique()))
print('Regions in which missing climate zones occur:', input_original[input_original['clim'].isin(set(input_original.clim.unique()) - set(input_nuts.clim.unique()))].region_bld.unique())

#checks
print('Missing values: ',input_nuts.isna().sum().sum())
print('Duplicates: ',input_nuts.duplicated().sum().sum())
print('NUTS3 regions: ',len(input_nuts.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(input_original[input_original.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True), input_nuts.drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

no double entries
Climate zones missing from EU-NUTS dataset: {'Zone_7B Very cold Dry', 'Zone_0B Extremely hot Dry', 'Zone_4B Mixed Dry', 'Zone_5B Cool Dry', 'Zone_6B Cold Dry', 'Zone_1B Very hot Dry'}
Regions in which missing climate zones occur: ['R32BRA' 'R32CAS-CAU' 'R32CAS-OTH' 'R32CHN' 'R32IND' 'R32MEA-H'
 'R32MEA-M' 'R32MEX' 'R32NAF' 'R32OAS-L-PAS' 'R32PAK' 'R32SSA-L'
 'R32SSA-M']
Missing values:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [9]:
    #Check which country-climatezone combinations only appear when having NUTS detail --> important to always use detailed clim file
    mapped = input_nuts.copy()

    # OLD CODE ONLY FOR COMPARISON! #TODO: remove this

    #import file
    filename= 'climatic_zones_rev'
    input = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

    #merge files
    input_nuts = pd.merge(input, region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), on=['region_bld'], how='right')

    duplic = input_nuts.copy()

    # Finding combinations that only appear in one of two dataframes

    # Get unique combinations for each dataframe
    duplic_combinations = set(zip(duplic['region_bld'], duplic['clim']))
    mapped_combinations = set(zip(mapped['region_bld'], mapped['clim']))

    # Find combinations that are only in duplic (not in mapped)
    only_in_duplic = duplic_combinations - mapped_combinations

    # Find combinations that are only in mapped (not in duplic)
    only_in_mapped = mapped_combinations - duplic_combinations

    # All combinations that don't appear in both
    not_in_both = only_in_duplic.union(only_in_mapped)

    print("Combinations only in duplic:")
    for combo in only_in_duplic:
        print(f"  region_bld: {combo[0]}, clim: {combo[1]}")

    print("\nCombinations only in mapped:")
    for combo in only_in_mapped:
        print(f"  region_bld: {combo[0]}, clim: {combo[1]}")

Combinations only in duplic:

Combinations only in mapped:
  region_bld: C-WEU-GRC, clim: Zone_4C Mixed Marine
  region_bld: C-WEU-PRT, clim: Zone_4A Mixed Humid
  region_bld: C-EEU-SVN, clim: Zone_4A Mixed Humid
  region_bld: C-WEU-FRA, clim: Zone_1A Very hot Humid
  region_bld: C-EEU-SVN, clim: Zone_7A Very cold Humid
  region_bld: C-EEU-LTU, clim: Zone_5A Cool Humid
  region_bld: C-WEU-FRA, clim: Zone_7A Very cold Humid
  region_bld: C-WEU-GRC, clim: Zone_5C Cool Marine
  region_bld: C-WEU-DEU, clim: Zone_6A Cold Humid
  region_bld: C-EEU-POL, clim: Zone_6A Cold Humid
  region_bld: C-WEU-ESP, clim: Zone_2B Hot Dry
  region_bld: C-WEU-SWE, clim: Zone_8A Subarctic/arctic Humid
  region_bld: C-WEU-AUT, clim: Zone_7A Very cold Humid
  region_bld: C-WEU-PRT, clim: Zone_4C Mixed Marine
  region_bld: C-WEU-ITA, clim: Zone_6A Cold Humid
  region_bld: C-WEU-IRL, clim: Zone_4A Mixed Humid
  region_bld: C-WEU-SWE, clim: Zone_7A Very cold Humid
  region_bld: C-WEU-GRC, clim: Zone_5A Cool Humid


In [104]:
#Check which files contain clim as index --> most files do
import os
import glob

folder_path = "/Users/Mira/Desktop/MCC/Code/EUBUCCO-IAM/messageix-buildings-subnational/2025_EU/input_resid"
files_with_clim = [os.path.basename(f) for f in glob.glob(os.path.join(folder_path, "*.csv")) if 'clim' in pd.read_csv(f, nrows=0).columns]
print("Files with 'clim' column:")
for filename in files_with_clim:
    print(f"  {filename}")

Files with 'clim' column:
  bld_shr_access_cool_resid_ssp2_rev.csv
  bld_share_mat_resid_ssp3_rev.csv
  bld_shr_access_cool_resid_ssp2_rev_nuts.csv
  bld_share_mat_resid_ssp2_rev.csv
  bld_shr_access_cool_resid_ssp3_rev.csv
  shr_need_heat_resid_rev.csv
  bld_shr_access_cool_resid_ssp1_rev.csv
  climatic_zones_rev_nuts.csv
  heat_operation_hours_ssp2_bld.csv
  bld_share_mat_resid_ssp1_rev.csv
  heat_intensity_LOW_rev.csv
  heat_intensity_rev_nuts.csv
  pop_clim_rev_SSP4.csv
  pop_clim_rev_SSP5.csv
  cool_intensity_rev_nuts.csv
  bld_share_mat_resid_ssp2_rev_nuts.csv
  shr_need_heat_resid_rev_nuts.csv
  pop_clim_rev_SSP1.csv
  cool_days_LOW_rev.csv
  shr_need_cool_resid_rev_nuts.csv
  heat_intensity_rev.csv
  pop_clim_rev_SSP2.csv
  cool_days_rev.csv
  pop_clim_rev_SSP3.csv
  cool_days.csv
  climatic_zones_rev.csv
  shr_need_cool_resid_rev.csv
  stock_baseyear_resid_rev_nuts.csv
  cool_days_rev_nuts.csv
  pop_clim_rev_SSP2_nuts.csv
  heat_operation_hours_ssp2.csv
  stock_baseyear_resid_

In [6]:
#Dwelling stock: urbanity, arch, climate zone, income class, year of construction, 

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWOB_R3
input = pd.read_csv('data/estat_cens_21dwob_r3_en.csv', usecols=['Housing', 'Type of building','geo', 'OBS_VALUE']) #dwellings by region and type of building
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['Housing'].isin(['Conventional dwellings', 'Unknown'])] #only occupied or non-occupied, unknown are 0
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch', 'Housing']).sum()
input = input.unstack().droplevel(0, axis=1)
input.columns.name = None
input = input.reset_index().rename({'geo':'NUTS-3 Code'}, axis=1)

input = nuts3code_to_region_nuts(input)
input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
input.set_index(['region_bld','region_nuts', 'clim', 'arch'], inplace=True)

#dwelling stock national distribution
filename = 'stock_baseyear_resid_rev'
input_trend_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

input_trend_original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]

#dwellings per person based on household sizes
hh = pd.read_csv('2025_EU/input_resid/hh_size_rev.csv')
hh = hh[hh.region_bld.isin(region_nuts_lab.region_bld.unique())][hh['year']==2020].set_index(list(hh.columns.drop('value'))) #cap/hhd
dwellings = input_trend_original.set_index(list(input_trend_original.columns.drop('value'))) 
input_trend_original = dwellings.div(dwellings.mul(hh)).reset_index() #dwellings/cap

#replace missing values in EUROSTAT with original model data
region_nuts_clim_year = pd.concat([region_nuts_clim]*len(input_trend_original.yr_con.unique()), keys=list(input_trend_original.yr_con.unique()), names=['yr_con', 'index']).reset_index().drop('index', axis=1)
region_nuts_clim_year = pd.concat([region_nuts_clim_year]*len(input_trend_original.inc_cl.unique()), keys=list(input_trend_original.inc_cl.unique()), names=['inc_cl', 'index']).reset_index().drop('index', axis=1)
region_nuts_clim_year = pd.concat([region_nuts_clim_year]*len(input_trend_original.arch.unique()), keys=list(input_trend_original.arch.unique()), names=['arch', 'index']).reset_index().drop('index', axis=1)

input_trend = pd.merge(input_trend_original, region_nuts_clim_year, how='right') #duplicating dwelling stock across same urt, clim, country; still 14 yr_con

input_trend[['mat']] = input_trend[['mat']].fillna(value='perm')
input_trend[['year']] = input_trend[['year']].fillna(value=2020)
replacement_map = dict(zip(input_trend_original[['yr_con', 'bld_age']].drop_duplicates().yr_con, input_trend_original[['yr_con', 'bld_age']].drop_duplicates().bld_age))
input_trend['bld_age'] = input_trend['bld_age'].fillna(input_trend.yr_con.map(replacement_map)) #TODO: impute this using the yr_con column

input_trend = input_trend.set_index(['region_bld', 'region_nuts', 'urt', 'clim', 'mat', 'arch', 'inc_cl','year', 'yr_con', 'bld_age']).drop('region_gea', axis=1)

#fill those regions for which no matching country-climate combinations were available, using simply the matching country-urbanity combination
input_trend_mean = input_trend.groupby(axis=0, level=[0,2,4,5,6,7,8]).mean() #country, urbanity, year used for filling nan values
filler_mapped = input_trend_mean.reindex(input_trend.index.droplevel([1, 3])) #nuts and climate not used
filler_mapped.index = input_trend.index
input_trend = input_trend.fillna(filler_mapped).reset_index()

input_trend = input_trend.dropna(how='any', axis=0) #remove insensible yr_con bld_age combinations

#multiply dwellings per capita by poopulation of broader category
pop = pd.read_csv('2025_EU/input_resid/pop_clim_rev_SSP2_nuts.csv')
pop = pop[pop['year']==2020]
input_trend = input_trend.set_index(list(input_trend.columns.drop('value'))).mul(pop.set_index(list(pop.columns.drop('value')))).reset_index()

#normalise estimated dwelling stock assuming equal shares of population across all nuts-urt combinations within each country-climate combination
nuts_in_reg = input_trend.drop('value', axis=1)[pop.columns.drop('value')].value_counts().reset_index() #number of segments within each pop combination
nuts_in_reg.columns = list(pop.columns)
input_trend = input_trend.set_index(list(input_trend.columns.drop('value'))).div(nuts_in_reg.set_index(list(nuts_in_reg.columns.drop('value')))).reset_index() #normalise population assuming equal shares of population across all nuts-urt combinations within each country-climate combination


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_69695/3267258012.py:27: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  hh = hh[hh.region_bld.isin(region_nuts_lab.region_bld.unique())][hh['year']==2020].set_index(list(hh.columns.drop('value'))) #cap/hhd


In [7]:
# Save the index to use for further processing
#set(column_list) - set(input_trend_original.columns) #{'eneff', 'region_nuts'}
idx_full = input_trend.drop(['value', 'year'], axis=1)
#adding energy efficiency levels to full index
idx_eneff = pd.read_csv('2025_EU/input_resid/heat_intensity_rev.csv')
idx_eneff = idx_eneff[idx_eneff.region_bld.isin(region_nuts_lab.region_bld.unique())]
idx_full = pd.concat([idx_full]*len(idx_eneff.eneff.unique()), keys=list(idx_eneff.eneff.unique()), names=['eneff', 'index']).reset_index().drop('index', axis=1)
#adding series of years to index
idx_year = pd.read_csv('2025_EU/input_resid/bld_shr_access_cool_resid_ssp2_rev.csv')
idx_year = idx_year[idx_year.region_bld.isin(region_nuts_lab.region_bld.unique())]
idx_full = pd.concat([idx_full]*len(idx_year.year.unique()), keys=list(idx_year.year.unique()), names=['year', 'index']).reset_index().drop('index', axis=1)
#add and align informal housing
idx_full.loc[idx_full['eneff']=='ns', 'arch'] = 'inf'
idx_full.loc[idx_full['eneff']=='ns', 'bld_age'] = 'ns'
idx_full.loc[idx_full['eneff']=='ns', 'mat'] = 'sub'
idx_full = idx_full.drop_duplicates()
idx_full['region_gea'] = idx_full.region_bld.str[2:5]

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'arch']).sum().drop(['year', 'yr_con'], axis=1)
input_trend_regurtarch.update(input.replace(0,np.NaN).rename({'Occupied conventional dwellings':'value'}, axis=1).drop('Unoccupied conventional dwellings', axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'clim', 'mat','arch', 'inc_cl', 'year', 'yr_con', 'bld_age']).div(input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'arch']).sum().drop(['year', 'yr_con'], axis=1))
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).reset_index()#.dropna(axis=0, how='all')



#match input trend out with construction period info
input_trend_out.replace({1965:1980, 1975:1980,
                         1970:1980,
                         1985:2000,
                         1990:2000, 
                         1995:2000,
                         2005:2010, 
                         #2015:2020,
                         }, inplace=True)
input_trend_out = input_trend_out.groupby(by=['region_bld','region_nuts','urt', 'clim', 'mat','arch', 'inc_cl', 'year', 'yr_con', 'bld_age']).sum()

#construction period
#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWOP_R3
input = pd.read_csv('data/estat_cens_21dwop_r3_en.csv', usecols=['Housing', 'y_const','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['Housing'].isin(['Conventional dwellings', 'Unknown'])] #only occupied or non-occupied, unknown are 0
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input = input.loc[~input['y_const'].isin(['UNK', 'TOTAL'])] #only direct numbers
input['y_const']=input.y_const.str[-4:]
input.replace({'2016':'2020', '1919':'1945'}, inplace=True)
input = input.groupby(by=['geo', 'y_const', 'Housing']).sum()
input = input.unstack().droplevel(0, axis=1)
input.columns.name = None
input.reset_index()
input = input.reset_index().rename({'geo':'NUTS-3 Code', 'y_const':'yr_con'}, axis=1)

#adding missing region labels
input = nuts3code_to_region_nuts(input)
input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
input['yr_con'] = input['yr_con'].astype(int)
input_shares = input.set_index(['region_bld','region_nuts', 'clim','yr_con']).div(input.groupby(by=['region_bld','region_nuts']).sum().drop('yr_con', axis=1)).reset_index() #TODO: - interpolate input_shares to more detailed years
input_shares['yr_con'] = input_shares['yr_con'].astype(int)
input_shares = input_shares.set_index(['region_bld','region_nuts', 'clim','yr_con']).unstack(3).reindex(region_nuts_clim.set_index(['region_bld','region_nuts', 'clim']).index).stack(dropna=False)

input_shares = input_shares.rename({'Occupied conventional dwellings':'value'}, axis=1).drop('Unoccupied conventional dwellings', axis=1)
input_shares = input_shares.reset_index().drop_duplicates()

#replace missing values in EUROSTAT with original model data
input_shares = input_shares.set_index(['region_bld','region_nuts', 'clim','yr_con']).fillna(input_trend_out.groupby(axis=0, level=[0,1,3,8]).sum().div(input_trend_out.groupby(axis=0, level=[0,1,3]).sum())) #TODO: this needs to be adapted to match the building years specified at EU scale

#merge construction age shares with input trend
input_trend_out = input_trend_out.groupby(axis=0, level=[0,1,2,3,4,5,6,7,9]).sum().multiply(input_shares).reset_index()

#input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out['region_gea'] = input_trend_out.region_bld.str[2:5]
input_trend_out['yr_con'] = input_trend_out['yr_con'].astype(int)
input_trend_out['year'] = input_trend_out['year'].astype(int)
input_trend_out.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original.copy()
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1)[original.region_bld.isin(region_nuts_lab.region_bld.unique())].sort_values('region_bld').reset_index(drop=True), \
                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))
#not all contained because of differing construction year

: 

In [7]:
idx_full.to_csv('/Volumes/KINGSTON/data/idx_full.csv')

In [ ]:

#NOTE: values for shr_need and bld_mat are only binary (0,1) in Europe in original files
def detail_other_inputs():
    # Non-specified Inputs Detailed: expanding tables to cover all NUTS regions
    column_list = ['region_nuts']
    input_list = pd.read_csv('2025_EU/input_list_resid_2025_06_27_nuts.csv')['SSP\xa02.00'].to_list()
    for filename in input_list:
        if filename not in ['regions_R61','climatic_zones_rev', 'stock_baseyear_resid_rev', 'pop_clim_rev_SSP2']: #for filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days_rev', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev', 'bld_shr_access_cool_resid_ssp2_rev', 'bld_share_mat_resid_ssp2_rev']:
            input = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')
            #for i in input.columns:
            #    if i not in column_list:
            #        column_list.append(i)
            if ('region_bld' in input.columns) and ('clim' in input.columns):   #there are two more files which have bld but not clim, households and floor space, those are disaggregated below
                input_cols = list(input.columns.drop('value'))
                print(input_cols)
                #merge files
                idx_merge = idx_full[input_cols + ['region_nuts']].drop_duplicates()
                input_nuts = pd.merge(input, idx_merge, on=input_cols, how='right')
                input_nuts_cols = list(input_nuts.columns.drop('value'))

                #impute values for missing country-climate zone combinations based on nearest neighbours
                df = input_nuts.set_index(input_nuts_cols)
                input_nuts = pd.DataFrame(KNNImputer(missing_values=np.nan, n_neighbors=3, weights='uniform')\
                            .fit(df).transform(df), index=df.index, columns=df.columns).reset_index()

                input_nuts['value'] = round(input_nuts.value, 8)

                #collect column labels
                for i in input_nuts_cols:
                    if i not in column_list:
                        column_list.append(i)
                
                #export
                input_nuts = input_nuts[~input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
                input_nuts.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

                #checks
                print(input_nuts_cols)
                original = input[input.region_bld.isin(region_nuts_lab.region_bld.unique())]
                out = input_nuts.copy()
                print('Missing values: ',out.isna().sum().sum())
                print('Null before: ',len(original[original.value==0]))
                print('Null after: ',len(out[out.value==0]))
                print('Duplicates: ',out.duplicated().sum().sum())
                print('NUTS3 regions: ',len(out.region_nuts.unique()))
                print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

                
                print(filename + ' DONE')
    print(column_list)
    
    return column_list

In [21]:
            filename = 'bld_share_mat_resid_ssp2_rev'
            input = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')
            #for i in input.columns:
            #    if i not in column_list:
            #        column_list.append(i)
            if ('region_bld' in input.columns) and ('clim' in input.columns):   #there are two more files which have bld but not clim, households and floor space, those are disaggregated below
                input_cols = list(input.columns.drop('value'))
                print(input_cols)
                #merge files
                idx_merge = idx_full[input_cols + ['region_nuts']].drop_duplicates()
                input_nuts = pd.merge(input, idx_merge, on=input_cols, how='right')
                input_nuts_cols = list(input_nuts.columns.drop('value'))

                #impute values for missing country-climate zone combinations based on nearest neighbours
                df = input_nuts.set_index(input_nuts_cols)
                input_nuts = pd.DataFrame(KNNImputer(missing_values=np.nan, n_neighbors=3, weights='uniform')\
                            .fit(df).transform(df), index=df.index, columns=df.columns).reset_index()
                
                input_nuts['value'] = round(input_nuts.value, 0)
                
                #export
                input_nuts = input_nuts[~input_nuts['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
                input_nuts.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

                #checks
                print(input_nuts_cols)
                original = input[input.region_bld.isin(region_nuts_lab.region_bld.unique())]
                out = input_nuts.copy()
                print('Missing values: ',out.isna().sum().sum())
                print('Null before: ',len(original[original.value==0]))
                print('Null after: ',len(out[out.value==0]))
                print('Duplicates: ',out.duplicated().sum().sum())
                print('NUTS3 regions: ',len(out.region_nuts.unique()))
                print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

                
                print(filename + ' DONE')

['region_bld', 'clim', 'urt', 'inc_cl', 'mat', 'year']
['region_bld', 'clim', 'urt', 'inc_cl', 'mat', 'year', 'region_nuts']
Missing values:  0
Null before:  5508
Null after:  132840
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True
bld_share_mat_resid_ssp2_rev DONE


In [35]:
# Household size: current

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWBNO_R3
input = pd.read_csv('data/estat_cens_21dwbno_r3_en.csv', usecols=['n_person', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input.replace({'GE11':'12'}, inplace=True) #aligning labels with model
input = input.loc[~input['n_person'].isin(['3-5', '6-10', 'GE6', 'TOTAL'])] #only direct numbers
input['population'] = input['n_person'].astype(float) * input['OBS_VALUE'] #calculate population per NUTS3 region based on number of occupants per dwelling and number of dwellings
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch']).sum()
input['hh_size'] = input['population'] / input['OBS_VALUE'] #calculate average household size per NUTS3 region and type of building based on population divided by number of dwellings
input = input.reset_index().drop(['OBS_VALUE'], axis=1).rename({'geo':'NUTS-3 Code'}, axis=1)

input = nuts3code_to_region_nuts(input)

input.set_index(['region_bld','region_nuts', 'arch'], inplace=True)

input_hh = input.copy()

# household size: trend
filename = 'hh_size_rev'
input_trend_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')
input_trend_original = pd.concat([input_trend_original.set_index(['region_bld', 'urt', 'year']), input_trend_original.set_index(['region_bld', 'urt', 'year'])], keys=['mfh', 'sfh'], names=['arch','region_bld', 'urt', 'year']).reorder_levels([1,2,0,3])

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(input_trend_original.reset_index(), region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), how='right')
input_trend['region_nuts'] = input_trend['region_nuts'].fillna(input_trend['region_bld'])

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean().drop('year', axis=1)
input_trend_regurtarch.update(input.replace(0,np.NaN).rename({'hh_size':'value'}, axis=1).drop('population', axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'arch', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'arch']).mean().drop('year', axis=1)) #QQ: why is there only hhd size trend in Croatia and no other EU countries?
#input_trend_diff = input_trend_diff.assign(value=1) #replacing Croatian trend in household sizes with stagnant household sizes
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() 

#export csv
input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original.reset_index()[input_trend_original.reset_index().region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_35791/1907105581.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  input = pd.read_csv('data/estat_cens_21dwbno_r3_en.csv', usecols=['n_person', 'Type of building','geo', 'OBS_VALUE'])
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_35791/1907105581.py:12: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input = input.groupby(by=['geo', 'arch']).sum()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_35791/1907105581.py:32: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_t

Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [11]:
# Floor area: current

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWBNR_R3
input = pd.read_csv('data/estat_cens_21dwbnr_r3_filtered_en.csv', usecols=['area', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['area'].isin(['UNK'])] #only direct numbers
input.replace({'SQM_LT30':20, 'SQM30-39':35,'SQM40-49':45,'SQM50-59':55,'SQM60-79':70,'SQM80-99':90, 'SQM100-119':110, 'SQM120-149':135, 'SQM_GE150':200}, inplace=True) #aligning labels with model
input['fa_total'] = input['area'].astype(float) * input['OBS_VALUE'] #calculate population per NUTS3 region based on number of occupants per dwelling and number of dwellings
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch']).sum()
input = input.reset_index().drop(['OBS_VALUE', 'area'], axis=1).rename({'geo':'NUTS-3 Code'}, axis=1)
input = nuts3code_to_region_nuts(input)
input.set_index(['region_bld','region_nuts', 'arch'], inplace=True)
input['fa_pc'] = input['fa_total'] / input_hh['population'] #calculate average household size per NUTS3 region and type of building based on population divided by number of dwellings

#fill up missing values in input['fa_pc']
# floor area: trend and missing values
filename = 'floor_resid_2024_12_27_REF'
input_trend_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(input_trend_original, region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1), how='right')

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'urt', 'arch']).mean().drop('year', axis=1)
input_trend_regurtarch.update(input.replace(0,np.NaN).rename({'fa_pc':'value'}, axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'mat','arch', 'eneff', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'urt', 'arch']).mean().drop('year', axis=1))
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() #QQ: why is there a twice as high floor arae per capita for the latest energy efficiency standards?

input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_35791/1359505202.py:27: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'urt', 'arch']).mean().drop('year', axis=1)
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_35791/1359505202.py:31: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'mat','arch', 'eneff', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld','region_nuts', 'urt', 'arch']).

Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


In [12]:
# Population: current
#import EUROSTAT projections: https://doi.org/10.2908/PROJ_19RP3
input = pd.read_csv('data/estat_proj_19rp3_filtered_en.csv', usecols=['geo', 'TIME_PERIOD','OBS_VALUE']) #residents per territory and year
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.rename({'geo':'NUTS-3 Code'}, axis=1)
input = nuts3code_to_region_nuts(input)
input = pd.merge(input, region_nuts_clim.drop('urt', axis=1).drop_duplicates()) #adding climate zone categories
input.rename({'TIME_PERIOD':'year', 'OBS_VALUE':'value'}, axis=1, inplace=True)
input = input[input['year'].isin(list(range(2020,2101,5)))]
input = (input.set_index(['region_bld', 'region_nuts', 'clim', 'year'])/1e6)#.reset_index() #million resident stock per year

#population national distribution
filename = 'pop_clim_rev_SSP2'
input_trend_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv') #million residents per country, urbanity, climate zone, and year

#merge index by duplicating values in original model data
region_nuts_clim_year = pd.concat([region_nuts_clim]*len(input_trend_original.year.unique()), keys=list(input_trend_original.year.unique()), names=['year', 'index']).reset_index().drop('index', axis=1)
input_trend = pd.merge(input_trend_original, region_nuts_clim_year, how='right') #duplicating population across same urt, clim, country
input_trend = input_trend.set_index(['region_bld', 'region_nuts', 'urt', 'clim', 'year'])

#fill those regions for which no matching country-climate combinations were available, using simply the matching country-urbanity combination
input_trend_mean = input_trend.groupby(axis=0, level=[0,2,4]).mean() #country, urbanity, year used for filling nan values
filler_mapped = input_trend_mean.reindex(input_trend.index.droplevel([1, 3])) #nuts and climate not used
filler_mapped.index = input_trend.index
input_trend = input_trend.fillna(filler_mapped).reset_index()

#normalise population assuming equal shares of population across all nuts-urt combinations within each country-climate combination
nuts_in_reg = region_nuts_clim.drop('urt', axis=1).drop_duplicates()[['region_bld', 'clim']].value_counts().reset_index() #number of nuts within each country-climate combination
nuts_in_reg.columns = ['region_bld', 'clim', 'value']
input_trend = input_trend.set_index(['region_bld','region_nuts', 'urt', 'clim', 'year']).div(nuts_in_reg.set_index(['region_bld', 'clim'])).reset_index() #normalise population assuming equal shares of population across all nuts-urt combinations within each country-climate combination

#include input values fom EUROSTAT
input_trend_regurtarch = input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum() #population per nuts and year
input_trend_regurtarch.update(input.replace(0,np.NaN)) #

#expand pop by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'clim', 'year']).div(input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum())
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() 

input_trend_out = input_trend_out[~input_trend_out['region_nuts'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

#checks
original = input_trend_original[input_trend_original.region_bld.isin(region_nuts_lab.region_bld.unique())]
out = input_trend_out.copy()
print('Missing values: ',out.isna().sum().sum())
print('Null before: ',len(original[original.value==0]))
print('Null after: ',len(out[out.value==0]))
print('Duplicates: ',out.duplicated().sum().sum())
print('NUTS3 regions: ',len(out.region_nuts.unique()))
print('Index still contained: ', all_rows_contained(original.drop('value', axis=1).sort_values('region_bld').reset_index(drop=True),\
                                                                    out.drop('value', axis=1).drop('region_nuts', axis=1).drop_duplicates().sort_values('region_bld').reset_index(drop=True)))

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_35791/1893316411.py:33: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_regurtarch = input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum() #population per nuts and year
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_35791/1893316411.py:37: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_diff = input_trend.set_index(['region_bld','region_nuts', 'urt', 'clim', 'year']).div(input_trend.groupby(by=['region_bld','region_nuts', 'clim', 'year']).sum())


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
NUTS3 regions:  1165
Index still contained:  True


## Increase resolution in aggregate level from world regions --> national

In [3]:
fa_pc = pd.read_csv('2025_EU/input_resid/floor_resid_2024_12_27_REF_nuts.csv')

In [5]:
fa_pc.value.describe()

count   358,820.00
mean         39.80
std          27.76
min          19.46
25%          28.60
50%          35.34
75%          49.96
max       3,065.75
Name: value, dtype: float64

In [3]:
input = pd.read_csv('2025_EU/input_resid/hh_size_rev_nuts.csv')

In [4]:
input

,region_bld,region_nuts,urt,arch,year,value
0,C-EEU-BGR,C-EEU-BGR-BG311,rur,mfh,2015,2.34
1,C-EEU-BGR,C-EEU-BGR-BG311,rur,mfh,2020,2.34
2,C-EEU-BGR,C-EEU-BGR-BG311,rur,mfh,2025,2.34
3,C-EEU-BGR,C-EEU-BGR-BG311,rur,mfh,2030,2.34
4,C-EEU-BGR,C-EEU-BGR-BG311,rur,mfh,2035,2.34
...,...,...,...,...,...,...
83875,C-WEU-SWE,C-WEU-SWE-SE332,urb,sfh,2080,2.34
83876,C-WEU-SWE,C-WEU-SWE-SE332,urb,sfh,2085,2.34
83877,C-WEU-SWE,C-WEU-SWE-SE332,urb,sfh,2090,2.34
83878,C-WEU-SWE,C-WEU-SWE-SE332,urb,sfh,2095,2.34


In [19]:
    input_list = pd.read_csv('2025_EU/input_list_resid_2025_06_27_nuts.csv')['SSP2-NUTS'].dropna().to_list()
    input_list = [i.removesuffix('_nuts') for i in input_list]
    for filename in input_list:
        if filename not in ['regions_R61','climatic_zones_rev', 'stock_baseyear_resid_rev', 'pop_clim_rev_SSP2']: #for filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days_rev', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev', 'bld_shr_access_cool_resid_ssp2_rev', 'bld_share_mat_resid_ssp2_rev']:
            input = pd.read_csv('2025_EU/input_resid/'+filename+'_nuts.csv')
            
            input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).mean().reset_index()
            input_nuts.to_csv('2025_EU/input_resid/'+filename+'_nuts_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['stock_baseyear_resid_rev', 'pop_clim_rev_SSP2']:
            input = pd.read_csv('2025_EU/input_resid/'+filename+'_nuts.csv')
            
            input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum().reset_index()
            input_nuts.to_csv('2025_EU/input_resid/'+filename+'_nuts_bld.csv', index=False)

            #checks
            original = input.copy()
            out = input_nuts.copy()
            print('Missing values: ',out.isna().sum().sum())
            print('Null before: ',len(original[original.value==0]))
            print('Null after: ',len(out[out.value==0]))
            print('Duplicates: ',out.duplicated().sum().sum())
                            
            print(filename + ' DONE')
        elif filename in ['regions_R61','climatic_zones_rev']:
            input = pd.read_csv('2025_EU/input_resid/'+filename+'_nuts.csv')
            
            input_nuts = input.drop('region_nuts', axis=1).drop_duplicates()
            input_nuts.to_csv('2025_EU/input_resid/'+filename+'_nuts_bld.csv', index=False)
                            
            print(filename + ' DONE')

regions_R61 DONE
climatic_zones_rev DONE
Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
pop_clim_rev_SSP2 DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_18430/1523838840.py:22: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum().reset_index()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_18430/1523838840.py:7: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).mean().reset_index()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
hh_size_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_18430/1523838840.py:7: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).mean().reset_index()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
floor_resid_2024_12_27_REF DONE


...

Missing values:  0
Null before:  706
Null after:  84
Duplicates:  0
cool_intensity_rev DONE
Missing values:  0
Null before:  238
Null after:  34
Duplicates:  0
cool_days_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_18430/1523838840.py:7: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).mean().reset_index()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_18430/1523838840.py:7: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).mean().reset_index()


Missing values:  0
Null before:  1381
Null after:  204
Duplicates:  0
shr_need_cool_resid_rev DONE
Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
shr_need_heat_resid_rev DONE


/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_18430/1523838840.py:22: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_nuts = input.groupby(by=list(input.columns.drop(['value', 'region_nuts'])), axis=0).sum().reset_index()


Missing values:  0
Null before:  0
Null after:  0
Duplicates:  0
stock_baseyear_resid_rev DONE


In [ ]:
input_list = pd.read_csv('2025_EU/input_list_resid_2025_06_27_nuts.csv')['SSP\xa02.00'].to_list()
for file in input_list:
    if file != 'regions_R61':
        input = pd.read_csv('2025_EU/input_resid/'+file+'.csv')
        if ('region_gea' in input.columns) and ('region_bld' not in input.columns):        
            #merge files
            input_nuts = pd.merge(input, region_lab, on=['region_gea'], how='right')
            #input_nuts = input_nuts.dropna(axis=0, how='any', subset='value')
            #input_nuts['region_bld'] = input_nuts['region_bld'].fillna(input_nuts['region_gea'])
            input_nuts.drop(['region_gea', 'R11', 'R12', 'iso3'], axis=1, inplace=True)

            #export
            #input_nuts = input_nuts[~input_nuts['region_bld'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
            input_nuts.to_csv('2025_EU/input_resid/'+file+'_bld.csv', index=False)

In [64]:
for file in input_list:
    df = pd.read_csv('2025_EU/input_resid/'+file+'.csv')
    if 'region_bld' in df.columns:
        print('bld ' + file)
    elif 'region_gea' in df.columns:
        print('gea ' + file)
    else:
        print(file + str(df.columns))

bld regions_R61
bld climatic_zones_rev
ct_bld_ageIndex(['bld_age_id', 'bld_age_name', 'mat', 'year_i', 'year_f'], dtype='object')
gea ct_bld
ct_eneffIndex(['eneff', 'mat', 'bld_age'], dtype='object')
ct_fuel_combIndex(['mat', 'fuel_heat', 'fuel_cool', 'mod_decision'], dtype='object')
ct_fuel_dhwIndex(['mat', 'fuel_heat', 'fuel_cool', 'res_solar', 'mod_decision'], dtype='object')
ct_ren_eneffIndex(['mat', 'eneff_i', 'eneff_f'], dtype='object')
ct_inc_clIndex(['urt', 'inc_cl'], dtype='object')
ct_tenrIndex(['mat', 'tenr'], dtype='object')
bld pop_clim_rev_SSP2
bld hh_size_rev
bld floor_resid_2024_12_27_REF
gea shr_hh_tenr
gea bld_shr_arch_resid
bld bld_share_mat_resid_ssp2_rev
gea bld_demolition_distr_long
bld bld_shr_access_cool_resid_ssp2_rev
gea bld_shr_district_heat_resid
gea bld_share_fuel_heat_resid_rev2024_02_21
gea material_int_resid
gea eff_heat_ssp2
gea eff_cool_ssp2_2024_12_27
gea eff_hotwater_resid
gea heat_floor_resid_rev2024_02_21
gea heat_operation_hours_ssp2
ren_energy_sa

## Replace Approach: replace region_bld labels with NUTS3 codes

In [ ]:
# Region Labels Detailed: creating labels that include an extra level for NUTS3 regions

#importing NUTS labels
nuts_lab = pd.read_excel('data/NUTS2021-NUTS2024.xlsx', sheet_name = 'NUTS-3 Typologies', header=0, usecols=['Country code', 'NUTS-3 Code', 'Urban-Rural typology'])
nuts_lab['iso3'] = coco.convert(names=nuts_lab['Country code'], to='ISO3')

#importing region labels
region_lab = pd.read_csv('2025_EU/input_others/regions_R61.csv')
region_lab['iso3']=region_lab.region_bld.str[-3:]

#appending NUTS labels to region labels where available
region_nuts_lab = pd.merge(nuts_lab, region_lab, on='iso3',how='outer')
region_nuts_lab['region_nuts'] = [str(x) + '_' + str(y) for x, y in zip(region_nuts_lab['region_bld'], region_nuts_lab['NUTS-3 Code'])]
region_nuts_lab['region_nuts'] = region_nuts_lab['region_nuts'].str.strip('_nan')

#cleaning df and harmonising with input dataframes
region_nuts_lab = region_nuts_lab.replace({'predominantly urban':'urb', 'intermediate':'urb', 'predominantly rural':'rur'})
region_nuts_lab = region_nuts_lab.rename({'Urban-Rural typology':'urt'}, axis=1)
region_nuts_lab_full = region_nuts_lab.copy()
region_nuts_lab = region_nuts_lab.drop(['iso3','NUTS-3 Code', 'Country code'], axis=1)

#adaptation for use to detail inputs
region_nuts_lab_idx = region_nuts_lab.drop(['region_gea', 'R11', 'R12'], axis=1)

#export
region_nuts_lab.drop(['region_bld','urt'], axis=1, inplace=True)
region_nuts_lab.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)
region_nuts_lab = region_nuts_lab.reindex(columns=region_lab.drop('iso3', axis=1).columns)
region_nuts_lab.to_csv('2025_EU/input_resid/regions_R61_nuts.csv', index=False)
region_nuts_lab.to_csv('2025_EU/input_others/regions_R61_nuts.csv', index=False)

In [67]:
len(region_nuts_lab.region_bld.unique())

1199

In [68]:
region_nuts_lab.iloc[0:1169].region_bld

0       C-WEU-BEL_BE100
1       C-WEU-BEL_BE211
2       C-WEU-BEL_BE212
3       C-WEU-BEL_BE213
4       C-WEU-BEL_BE223
             ...       
1164    C-WEU-SWE_SE332
1165          C-WEU-CHE
1166          C-WEU-GBR
1167          C-WEU-ISL
1168          C-WEU-NOR
Name: region_bld, Length: 1169, dtype: object

In [69]:
    #import file
    filename= 'climatic_zones_rev'
    input = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

    #merge files
    input_nuts = pd.merge(input, region_nuts_lab_idx, on=['region_bld', 'urt'], how='outer')
    input_nuts = input_nuts.dropna(axis=0, how='any', subset='clim')
    input_nuts['region_nuts'] = input_nuts['region_nuts'].fillna(input_nuts['region_bld'])
    input_nuts.drop('region_bld', axis=1, inplace=True)
    input_nuts.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)
    input_nuts = input_nuts.reindex(columns=input.columns)

    #export
    input_nuts = input_nuts[~input_nuts['region_bld'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
    input_nuts.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

In [70]:
# Non-specified Inputs Detailed: expanding tables to cover all NUTS regions

for filename in ['heat_intensity_rev', 'cool_intensity_rev', 'cool_days_rev', 'shr_need_cool_resid_rev', 'shr_need_heat_resid_rev', 'bld_shr_access_cool_resid_ssp2_rev', 'bld_share_mat_resid_ssp2_rev']:
    #import file
    input = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

    #merge files
    input_nuts = pd.merge(input, region_nuts_lab_idx, on=['region_bld', 'urt'], how='outer')
    input_nuts = input_nuts.dropna(axis=0, how='any', subset='value')
    input_nuts['region_nuts'] = input_nuts['region_nuts'].fillna(input_nuts['region_bld'])
    input_nuts.drop('region_bld', axis=1, inplace=True)
    input_nuts.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)
    input_nuts = input_nuts.reindex(columns=input.columns)

    #export
    input_nuts = input_nuts[~input_nuts['region_bld'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
    input_nuts.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

In [24]:
set(input_nuts.region_bld.unique())-set(region_nuts_lab.region_bld.unique())

{'C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'}

In [ ]:
region_nuts_lab.region_bld.unique()

In [71]:
len(input_nuts.region_bld.unique())

1199

In [72]:
# Household size: current

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWBNO_R3
input = pd.read_csv('data/estat_cens_21dwbno_r3_en.csv', usecols=['n_person', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input.replace({'GE11':'12'}, inplace=True) #aligning labels with model
input = input.loc[~input['n_person'].isin(['3-5', '6-10', 'GE6', 'TOTAL'])] #only direct numbers
input['population'] = input['n_person'].astype(float) * input['OBS_VALUE'] #calculate population per NUTS3 region based on number of occupants per dwelling and number of dwellings
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch']).sum()
input['hh_size'] = input['population'] / input['OBS_VALUE'] #calculate average household size per NUTS3 region and type of building based on population divided by number of dwellings
input = input.reset_index().drop(['OBS_VALUE'], axis=1).rename({'geo':'NUTS-3 Code'}, axis=1)
input = pd.merge(input,region_nuts_lab_full[['NUTS-3 Code', 'urt', 'region_nuts', 'region_bld']]) #adding region labels
input.drop(['region_bld', 'NUTS-3 Code'], axis=1, inplace=True)
input.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)
input.set_index(['region_bld', 'urt', 'arch'], inplace=True)

input_hh = input.copy()

# household size: trend
filename = 'hh_size_rev'
input_trend_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')
input_trend_original = pd.concat([input_trend_original.set_index(['region_bld', 'urt', 'year']), input_trend_original.set_index(['region_bld', 'urt', 'year'])], keys=['mfh', 'sfh'], names=['arch','region_bld', 'urt', 'year']).reorder_levels([1,2,0,3])

#fill up missing values in input

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(region_nuts_lab_idx, input_trend_original.reset_index(), how='outer')
input_trend['region_nuts'] = input_trend['region_nuts'].fillna(input_trend['region_bld'])
input_trend.drop('region_bld', axis=1, inplace=True)
input_trend.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld', 'urt', 'arch']).mean().drop('year', axis=1)
input_trend_regurtarch.update(input.replace(0,np.NaN).rename({'hh_size':'value'}, axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld', 'urt', 'arch', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld', 'urt', 'arch']).mean().drop('year', axis=1))
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() #QQ: why is there a twice as high floor arae per capita for the latest energy efficiency standards?

#export csv
input_trend_out = input_trend_out[~input_trend_out['region_bld'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_36030/4165527250.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  input = pd.read_csv('data/estat_cens_21dwbno_r3_en.csv', usecols=['n_person', 'Type of building','geo', 'OBS_VALUE'])
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_36030/4165527250.py:12: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input = input.groupby(by=['geo', 'arch']).sum()


In [73]:
len(input_trend_out.region_bld.unique())

1199

In [74]:
# Population: current
#import EUROSTAT projections: https://doi.org/10.2908/PROJ_19RP3
input = pd.read_csv('data/estat_proj_19rp3_filtered_en.csv', usecols=['geo', 'TIME_PERIOD','OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level

input = input.rename({'geo':'NUTS-3 Code'}, axis=1)
input = pd.merge(input,region_nuts_lab_full[['NUTS-3 Code', 'urt', 'region_nuts', 'region_bld']]) #adding region labels
input.drop(['region_bld', 'NUTS-3 Code'], axis=1, inplace=True)
input.rename({'region_nuts':'region_bld', 'TIME_PERIOD':'year', 'OBS_VALUE':'value'}, axis=1, inplace=True)
input = input[input['year'].isin(list(range(2020,2101,5)))]
input = (input.set_index(['region_bld', 'urt', 'year'])/1e6).reset_index()

#population national distribution
filename = 'pop_clim_rev_SSP2'
input_trend_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(region_nuts_lab_idx, input_trend_original, how='outer')
input_trend['region_nuts'] = input_trend['region_nuts'].fillna(input_trend['region_bld'])
input_trend.drop('region_bld', axis=1, inplace=True)
input_trend.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)

#include input values fom EUROSTAT
input_trend_regurtarch = input_trend.groupby(by=['region_bld', 'urt', 'year']).sum()
input_trend_regurtarch.update(input.replace(0,np.NaN))

#expand pop by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld', 'urt', 'clim', 'year']).div(input_trend.groupby(by=['region_bld', 'urt', 'year']).sum())
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() #QQ: why is there a twice as high floor arae per capita for the latest energy efficiency standards?

input_trend_out = input_trend_out[~input_trend_out['region_bld'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_36030/4161653047.py:24: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_regurtarch = input_trend.groupby(by=['region_bld', 'urt', 'year']).sum()
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_36030/4161653047.py:28: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_diff = input_trend.set_index(['region_bld', 'urt', 'clim', 'year']).div(input_trend.groupby(by=['region_bld', 'urt', 'year']).sum())


In [75]:
len(input_trend_out.region_bld.unique())

1199

In [76]:
# Floor area: current

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWBNR_R3
input = pd.read_csv('data/estat_cens_21dwbnr_r3_filtered_en.csv', usecols=['area', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['area'].isin(['UNK'])] #only direct numbers
input.replace({'SQM_LT30':20, 'SQM30-39':35,'SQM40-49':45,'SQM50-59':55,'SQM60-79':70,'SQM80-99':90, 'SQM100-119':110, 'SQM120-149':135, 'SQM_GE150':200}, inplace=True) #aligning labels with model
input['fa_total'] = input['area'].astype(float) * input['OBS_VALUE'] #calculate population per NUTS3 region based on number of occupants per dwelling and number of dwellings
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch']).sum()
input = input.reset_index().drop(['OBS_VALUE', 'area'], axis=1).rename({'geo':'NUTS-3 Code'}, axis=1)
input = pd.merge(input,region_nuts_lab_full[['NUTS-3 Code', 'urt', 'region_nuts', 'region_bld']]) #adding region labels
input.drop(['region_bld', 'NUTS-3 Code'], axis=1, inplace=True)
input.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)
input.set_index(['region_bld', 'urt', 'arch'], inplace=True)
input['fa_pc'] = input['fa_total'] / input_hh['population'] #calculate average household size per NUTS3 region and type of building based on population divided by number of dwellings

#fill up missing values in input['fa_pc']
# floor area: trend and missing values
filename = 'floor_resid_2024_12_27_REF'
input_trend_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(region_nuts_lab_idx, input_trend_original, how='outer')
input_trend['region_nuts'] = input_trend['region_nuts'].fillna(input_trend['region_bld'])
input_trend.drop('region_bld', axis=1, inplace=True)
input_trend.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld', 'urt', 'arch']).mean().drop('year', axis=1)
input_trend_regurtarch.update(input.replace(0,np.NaN).rename({'fa_pc':'value'}, axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld', 'urt', 'mat','arch', 'eneff', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld', 'urt', 'arch']).mean().drop('year', axis=1))
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() #QQ: why is there a twice as high floor arae per capita for the latest energy efficiency standards?

input_trend_out = input_trend_out[~input_trend_out['region_bld'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_36030/3062169772.py:32: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_regurtarch = input_trend[input_trend['year']==2020].groupby(by=['region_bld', 'urt', 'arch']).mean().drop('year', axis=1)
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_36030/3062169772.py:36: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_diff = input_trend.set_index(['region_bld', 'urt', 'mat','arch', 'eneff', 'year']).div(input_trend[input_trend['year']==2020].groupby(by=['region_bld', 'urt', 'arch']).mean().drop('year', axis=1))


In [176]:
len(input_trend_out.region_bld.unique())

1199

In [77]:
#Dwelling stock: urbanity, arch, climate zone, income class, year of construction, 

#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWOB_R3
input = pd.read_csv('data/estat_cens_21dwob_r3_en.csv', usecols=['Housing', 'Type of building','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['Housing'].isin(['Conventional dwellings', 'Unknown'])] #only occupied or non-occupied, unknown are 0
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input.rename({'Type of building':'arch'}, axis=1, inplace=True)
input = input[input['arch'].isin(['mfh', 'sfh'])]
input = input.groupby(by=['geo', 'arch', 'Housing']).sum()
input = input.unstack().droplevel(0, axis=1)
input.columns.name = None
input.reset_index()
input = input.reset_index().rename({'geo':'NUTS-3 Code'}, axis=1)
input = pd.merge(input,region_nuts_lab_full[['NUTS-3 Code', 'urt', 'region_nuts']]) #adding region labels
input.drop(['NUTS-3 Code'], axis=1, inplace=True)
#input.rename({'region_bld':'country','region_nuts':'region_bld'}, axis=1, inplace=True)
input.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)
input.set_index(['region_bld', 'urt', 'arch'], inplace=True)

#dwelling stock national distribution
filename = 'stock_baseyear_resid_rev'
input_trend_original = pd.read_csv('2025_EU/input_resid/'+filename+'.csv')

#replace missing values in EUROSTAT with original model data
input_trend = pd.merge(region_nuts_lab_idx, input_trend_original, how='outer')
input_trend['region_nuts'] = input_trend['region_nuts'].fillna(input_trend['region_bld'])
input_trend.drop('region_bld', axis=1, inplace=True)
input_trend.rename({'region_nuts':'region_bld'}, axis=1, inplace=True)

#include input['fa_pc'] values fom EUROSTAT
input_trend_regurtarch = input_trend.groupby(by=['region_bld', 'urt', 'arch']).sum().drop(['year', 'yr_con'], axis=1)
input_trend_regurtarch.update(input.replace(0,np.NaN).rename({'Occupied conventional dwellings':'value'}, axis=1).drop('Unoccupied conventional dwellings', axis=1))

#expand fa_pc by using distribution of values from input_trend_original
input_trend_diff = input_trend.set_index(['region_bld', 'region_gea','urt', 'clim', 'mat','arch', 'inc_cl', 'year', 'yr_con', 'bld_age']).div(input_trend.groupby(by=['region_bld', 'urt', 'arch']).sum().drop(['year', 'yr_con'], axis=1))
input_trend_out = input_trend_diff.multiply(input_trend_regurtarch).dropna(axis=0, how='all').reset_index() #QQ: why is there a twice as high floor arae per capita for the latest energy efficiency standards?

input_trend_out = input_trend_out[~input_trend_out['region_bld'].isin(['C-WEU-CYP', 'C-WEU-LUX', 'C-WEU-MLT'])]
input_trend_out.to_csv('2025_EU/input_resid/'+filename+'_nuts.csv', index=False)

/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_36030/1672794917.py:32: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_regurtarch = input_trend.groupby(by=['region_bld', 'urt', 'arch']).sum().drop(['year', 'yr_con'], axis=1)
/var/folders/__/twfrdq8n75n5f0118_f882700000gn/T/ipykernel_36030/1672794917.py:36: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  input_trend_diff = input_trend.set_index(['region_bld', 'region_gea','urt', 'clim', 'mat','arch', 'inc_cl', 'year', 'yr_con', 'bld_age']).div(input_trend.groupby(by=['region_bld', 'urt', 'arch']).sum().drop(['year', 'yr_con'], axis=1))


In [178]:
len(input_trend_out.region_bld.unique())

1199

In [78]:
#construction period
#import the EUROSTAT census dataset: https://doi.org/10.2908/CENS_21DWOP_R3
input = pd.read_csv('data/estat_cens_21dwop_r3_en.csv', usecols=['Housing', 'y_const','geo', 'OBS_VALUE'])
input = input.loc[input['geo'].str.len() == 5] #only NUTS3 level
input = input.loc[~input['Housing'].isin(['Conventional dwellings', 'Unknown'])] #only occupied or non-occupied, unknown are 0
input.replace({'One-dwelling residential buildings':'sfh', 'Three or more dwelling residential buildings':'mfh', 'Two-dwelling residential buildings':'mfh', 'Non-residential buildings':'mfh'}, inplace=True) #aligning labels with model
input = input.loc[~input['y_const'].isin(['UNK', 'TOTAL'])] #only direct numbers
input['y_const']=input.y_const.str[-4:]
input.replace({'2016':'2020'}, inplace=True)
input = input.groupby(by=['geo', 'y_const', 'Housing']).sum()
input = input.unstack().droplevel(0, axis=1)
input.columns.name = None
input.reset_index()
input = input.reset_index().rename({'geo':'NUTS-3 Code', 'y_const':'yr_con'}, axis=1)
input = pd.merge(input,region_nuts_lab_full[['NUTS-3 Code', 'urt', 'region_nuts', 'region_bld']]) #adding region labels
input.drop(['NUTS-3 Code'], axis=1, inplace=True)
input.rename({'region_bld':'country','region_nuts':'region_bld'}, axis=1, inplace=True)
input.set_index(['country','region_bld', 'yr_con', 'urt'], inplace=True)

In [440]:
input

Occupied conventional dwellings  \
country   region_bld      yr_con urt                                    
C-WEU-AUT C-WEU-AUT_AT111 1919   rur                              932   
                          1945   rur                             1051   
                          1960   rur                             1525   
                          1980   rur                             5232   
                          2000   rur                             3807   
...                                                               ...   
C-EEU-SVK C-EEU-SVK_SK042 1980   urb                           103465   
                          2000   urb                            54233   
                          2010   urb                             7466   
                          2015   urb                             4207   
                          2020   urb                             3822   

                                      Unoccupied conventional dwellings  
country   region_bld      yr_con urt                                     
C-WEU-AUT C-WEU-AUT_AT111 1919   rur                                867  
                          1945   rur                                746  
                          1960   rur                                816  
                          1980   rur                               1646  
                          2000   rur                                682  
...                                                                 ...  
C-EEU-SVK C-EEU-SVK_SK042 1980   urb                              14833  
                          2000   urb                               5035  
                          2010   urb                               1855  
                          2015   urb                               1627  
                          2020   urb                               3586  

[8928 rows x 2 columns]

In [ ]:
# Tenure: current